<a href="https://colab.research.google.com/github/ONOSPETER/DevWitness/blob/main/devwitness_walrus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 DevWitness — Persistent AI Memory Agent on Walrus

---

## What This Notebook Demonstrates

**DevWitness** is a technical memory agent that proves one core idea:

> *Without memory, every AI session starts blind. With Walrus Memory, decisions persist on-chain across sessions, across AI providers, and across time.*

---

## Architecture

```
┌─────────────────────────────────────────────────────┐
│                   USER (CLI)                        │
│                      │                              │
│              ┌───────▼────────┐                     │
│              │  SYSTEM PROMPT │  ← The soul of the  │
│              │   (editable)   │    agent. Controls  │
│              └───────┬────────┘    all behavior.    │
│                      │                              │
│         ┌────────────▼─────────────┐                │
│         │    WALRUS MEMORY LAYER   │                │
│         │  recall() → inject past  │                │
│         │  remember() → save new   │                │
│         └────────────┬─────────────┘                │
│                      │                              │
│    ┌─────────────────▼──────────────────────┐       │
│    │           AI PROVIDER ROUTER           │       │
│    │  1st: Gemini  2nd: Groq  3rd: ChatGPT  │       │
│    └────────────────────────────────────────┘       │
└─────────────────────────────────────────────────────┘
```

## The Role of the Prompt

The **system prompt** is not just configuration — it IS the agent's intelligence layer.
It tells the AI:
- When to signal a memory write (`[REMEMBER]:`)
- How to surface recalled memories (`[RECALLED]`)
- How to behave when memory is absent (`[NO PRIOR MEMORY]`)

Change the prompt → change the agent's entire personality and memory behavior.
This is why the Settings menu lets you edit it live.

## AI Provider Strategy

| Priority | Provider | Model | Why |
|---|---|---|---|
| 1st | **Gemini** | gemini-2.0-flash | Default — fast, free tier generous |
| 2nd | **Groq** | llama-3.3-70b | Fallback — extremely fast inference |
| 3rd | **ChatGPT** | gpt-4o-mini | Final fallback — reliable, wider availability |

Memory and prompt persist identically across all three. The AI changes. The memory does not.

---

**Run cells in order from top to bottom.**

## Cell 1 — Install Dependencies

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Install all required packages
# Run this once per Colab session
# ─────────────────────────────────────────────

!pip install -q memwal nest_asyncio google-genai groq openai

print("✅ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.5 MB/s eta 0:00:00
✅ All packages installed


## Cell 2 — API Keys & Configuration

Fill in your keys below. In Colab, you can also use **Secrets** (🔑 icon in left sidebar)
and the code will read from there automatically.

**Required:**
- `GEMINI_API_KEY` → [aistudio.google.com](https://aistudio.google.com)
- `WALRUS_DELEGATE_KEY` → [memory.walrus.xyz](https://memory.walrus.xyz)
- `WALRUS_ACCOUNT_ID` → same dashboard
- `WALRUS_RELAYER_URL` → same dashboard

**Optional (for fallback):**
- `GROQ_API_KEY` → [console.groq.com](https://console.groq.com)
- `OPENAI_API_KEY` → [platform.openai.com](https://platform.openai.com)

In [12]:
# ─────────────────────────────────────────────
# CELL 2: Configuration
# ─────────────────────────────────────────────

import os
import nest_asyncio
nest_asyncio.apply()  # Allow async in Colab's existing event loop
from google.colab import userdata
#userdata.get('secretName')

try:
    from google.colab import userdata
    def get_secret(key, default=""):
        try:
            return userdata.get(key) or default
        except:
            return os.environ.get(key, default)
except ImportError:
    def get_secret(key, default=""):
        return os.environ.get(key, default)


CONFIG = {
    # ── AI Keys ──
    "GEMINI_API_KEY":    get_secret("GEMINI_API_KEY",   userdata.get("GEMINI_API_KEY")),
    "GROQ_API_KEY":      get_secret("GROQ_API_KEY",     userdata.get("GROQ_API_KEY")),
    "OPENAI_API_KEY":    get_secret("OPENAI_API_KEY",   userdata.get("OPENAI_API_KEY")),

    # ── MemWal Keys (official variable names from docs) ──
    "MEMWAL_PRIVATE_KEY": get_secret("MEMWAL_PRIVATE_KEY", userdata.get("MEMWAL_PRIVATE_KEY")),
    "MEMWAL_ACCOUNT_ID":  get_secret("MEMWAL_ACCOUNT_ID",  userdata.get("MEMWAL_ACCOUNT_ID")),
    "MEMWAL_SERVER_URL":  get_secret("MEMWAL_SERVER_URL",
                                     "https://relayer.memory.walrus.xyz"),
    "NAMESPACE": "devwitness",
}

def check_config():
    print("\n╔══════════════════════════════════╗")
    print(  "║     DEVWITNESS CONFIG CHECK      ║")
    print(  "╚══════════════════════════════════╝")
    checks = {
        "Gemini":          CONFIG["GEMINI_API_KEY"],
        "Groq":            CONFIG["GROQ_API_KEY"],
        "OpenAI":          CONFIG["OPENAI_API_KEY"],
        "MemWal Key":      CONFIG["MEMWAL_PRIVATE_KEY"],
        "MemWal Account":  CONFIG["MEMWAL_ACCOUNT_ID"],
    }
    for k, v in checks.items():
        icon = "✅" if v else "❌ Missing"
        print(f"  {k:<18} {icon}")
    print(f"\n  Relayer: {CONFIG['MEMWAL_SERVER_URL']}\n")

check_config()


╔══════════════════════════════════╗
║     DEVWITNESS CONFIG CHECK      ║
╚══════════════════════════════════╝
  Gemini             ✅
  Groq               ✅
  OpenAI             ✅
  MemWal Key         ✅
  MemWal Account     ✅

  Relayer: https://relayer.memory.walrus.xyz



## Cell 3 — Walrus Memory Layer

Two core functions:
- `walrus_remember(text)` — writes a blob to Walrus Mainnet, returns job_id
- `walrus_recall(query)` — retrieves semantically relevant past memories

These are the only two functions the agent needs to interact with persistent memory.
All authentication happens via request headers — no wallet popup required.

In [13]:
# ─────────────────────────────────────────────
# CELL 3: Walrus Memory Layer
# ─────────────────────────────────────────────

import asyncio
from memwal import MemWal

# ── Singleton SDK instance ──
_memwal_client = None


async def _get_client():
    """
    Create or return the MemWal SDK client.
    The client is reused across calls — creating it once per session.
    """
    global _memwal_client
    if _memwal_client is None:
        _memwal_client = MemWal.create(
            key=CONFIG["MEMWAL_PRIVATE_KEY"],
            account_id=CONFIG["MEMWAL_ACCOUNT_ID"],
            server_url=CONFIG["MEMWAL_SERVER_URL"],
        )
    return _memwal_client


def walrus_remember(decision_text: str) -> dict:
    """
    Write a memory blob to Walrus via the official MemWal Python SDK.

    SDK call:
        await memwal.remember_and_wait(text)
    Handles internally:
        embedding → SEAL encryption → Walrus upload → vector index
    """
    if not CONFIG["MEMWAL_PRIVATE_KEY"]:
        return {"status": "skipped", "reason": "No MemWal key configured"}

    from datetime import datetime, timezone
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    content   = f"[{timestamp}] DECISION: {decision_text}"

    async def _remember():
        client = await _get_client()
        result = await client.remember_and_wait(content)
        return result

    try:
        loop   = asyncio.get_event_loop()
        result = loop.run_until_complete(_remember())
        # result is the blob/job object — extract ID
        blob_id = getattr(result, "blob_id", None) or getattr(result, "id", None) or str(result)
        print(f"  [remember] blob_id={blob_id}")
        return {"status": "saved", "job_id": blob_id}
    except Exception as e:
        print(f"  [remember ERROR] {e}")
        return {"status": "error", "error": str(e)}


def walrus_recall(query: str, limit: int = 4) -> list:
    """
    Retrieve relevant memories from Walrus via the official MemWal Python SDK.

    SDK call:
        result = await memwal.recall(query)
        result.results[0].text
    """
    if not CONFIG["MEMWAL_PRIVATE_KEY"]:
        return []

    async def _recall():
        client = await _get_client()
        result = await client.recall(query)
        return result

    try:
        loop   = asyncio.get_event_loop()
        result = loop.run_until_complete(_recall())
        # Official response: result.results → list of objects with .text
        items  = getattr(result, "results", [])
        texts  = [getattr(r, "text", "") for r in items if getattr(r, "text", "")]
        print(f"  [recall] returned {len(texts)} memories")
        return texts
    except Exception as e:
        print(f"  [recall ERROR] {e}")
        return []


def walrus_blob_count() -> int:
    """Estimate blob count using a broad recall query."""
    try:
        loop   = asyncio.get_event_loop()
        async def _count():
            client = await _get_client()
            result = await client.recall("decision")
            return getattr(result, "total", len(getattr(result, "results", [])))
        return loop.run_until_complete(_count())
    except Exception:
        return -1


# ── CONNECTION TEST ──
print("=" * 50)
print("  WALRUS CONNECTION TEST")
print("=" * 50)
print(f"  Relayer : {CONFIG['MEMWAL_SERVER_URL']}")
acct = CONFIG["MEMWAL_ACCOUNT_ID"]
print(f"  Account : {acct[:12]}...{acct[-6:]}" if len(acct) > 18 else f"  Account : {acct or 'NOT SET'}")
print()

if CONFIG["MEMWAL_PRIVATE_KEY"]:
    print("  Testing remember...")
    w = walrus_remember("TEST: Colab connection verified — DevWitness online")
    print(f"  Result : {w}")

    print("\n  Testing recall...")
    mems = walrus_recall("connection test")
    print(f"  Memories found : {len(mems)}")
    if mems:
        print(f"  Sample : {mems[0][:80]}")
else:
    print("  ❌ MEMWAL_PRIVATE_KEY not set — add it to Colab Secrets")

print("\n✅ Walrus Memory layer ready")

  WALRUS CONNECTION TEST
  Relayer : https://relayer.memory.walrus.xyz
  Account : 0x444aa07745...39698d

  Testing remember...
  [remember] blob_id=bzPUp2st3L-Tw4VXKN7uAmfW9HS7R044f0dj_8Vuntc
  Result : {'status': 'saved', 'job_id': 'bzPUp2st3L-Tw4VXKN7uAmfW9HS7R044f0dj_8Vuntc'}

  Testing recall...
  [recall] returned 10 memories
  Memories found : 10
  Sample : [2026-07-04 12:13 UTC] DECISION: TEST: Colab connection verified — DevWitness on

✅ Walrus Memory layer ready


## Cell 4 — AI Provider Router

The router tries providers in priority order:
1. **Gemini** (Google AI Studio — default)
2. **Groq** (llama-3.3-70b — fast fallback)
3. **ChatGPT** (gpt-4o-mini — reliable final fallback)

The same `system_prompt` and injected `memory_block` are sent to all three.
This proves that **memory and prompt are provider-agnostic** — only the inference engine changes.

In [14]:
# ─────────────────────────────────────────────
# CELL 4: AI Provider Router
# ─────────────────────────────────────────────

import requests as _requests
from groq import Groq
from openai import OpenAI
from google import genai
from google.genai import types as genai_types


def call_gemini(prompt: str) -> str:
    """Call Gemini 2.0 Flash via new google-genai SDK."""
    client = genai.Client(api_key=CONFIG["GEMINI_API_KEY"])
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt,
    )
    return response.text


def call_groq(prompt: str) -> str:
    """Call Llama 3.3 70B via Groq."""
    client = Groq(api_key=CONFIG["GROQ_API_KEY"])
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
    )
    return completion.choices[0].message.content


def call_openai(prompt: str) -> str:
    """Call GPT-4o Mini via OpenAI."""
    client = OpenAI(api_key=CONFIG["OPENAI_API_KEY"])
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
    )
    return completion.choices[0].message.content


PROVIDERS = {
    "gemini": {"fn": call_gemini,  "name": "Gemini 2.0 Flash",      "key": "GEMINI_API_KEY"},
    "groq":   {"fn": call_groq,    "name": "Groq / Llama-3.3-70B",  "key": "GROQ_API_KEY"},
    "openai": {"fn": call_openai,  "name": "ChatGPT / GPT-4o Mini", "key": "OPENAI_API_KEY"},
}

PROVIDER_ORDER = ["gemini", "groq", "openai"]


def call_ai(prompt: str, preferred: str = "gemini") -> tuple:
    order = [preferred] + [p for p in PROVIDER_ORDER if p != preferred]
    for provider_key in order:
        provider = PROVIDERS[provider_key]
        if not CONFIG.get(provider["key"]):
            continue
        try:
            response = provider["fn"](prompt)
            return response, provider["name"]
        except Exception as e:
            print(f"  ⚠️  {provider['name']} failed: {str(e)[:80]}")
            print(f"  ↪️  Trying next provider...")
            continue
    raise RuntimeError("All AI providers failed. Check your API keys.")


print("✅ AI Provider Router loaded (google-genai SDK)")

✅ AI Provider Router loaded (google-genai SDK)


## Cell 5 — Agent Core

The agent core does three things per turn:
1. **Recall** — fetch relevant memories from Walrus and inject into prompt
2. **Respond** — send the enriched prompt to the active AI provider
3. **Remember** — if the response signals a decision, write a blob to Walrus

The `[REMEMBER]:` signal in the response is how the prompt instructs the AI
to trigger a memory write. No signal = no write. The prompt controls everything.

In [ ]:
import re

# ── Default system prompt — the agent's soul ──
DEFAULT_SYSTEM_PROMPT = """You are DevWitness, a persistent technical memory agent for developers.
Your job is to ensure no technical decision is made twice without awareness of the first time it was made.

Every turn:
1. If memory context is provided above, reference it and start with [RECALLED]
2. If the user makes a technical decision (library, database, architecture,
   tool choice, API design), end your response with:
   [REMEMBER]: <one clear sentence summarizing the decision made>
3. If no prior memory exists on the topic, start with [NO PRIOR MEMORY]
4. Confirm every Walrus save with: Logged to Walrus

Keep responses conversational and informative, but concise, aiming for proper dialogue rather than just one or two-line sentences. Be direct, sharp, and analytical.
When memory is present, reference it explicitly — say what was decided
before and whether the current decision aligns or contradicts it."""


# ── Mutable agent state ──
AGENT_STATE = {
    "system_prompt":      DEFAULT_SYSTEM_PROMPT,
    "preferred_provider": "gemini",
    "memory_enabled":     True,
    "session_blob_count": 0,
    "total_turns":        0,
}


def extract_remember_signal(text: str):
    """
    Parse [REMEMBER]: signal from AI response.

    The system prompt instructs the AI to append [REMEMBER]: followed
    by a one-sentence decision summary when a technical choice is made.
    This function extracts that sentence to write to Walrus.

    Returns:
        str or None: The decision sentence, or None if not present.
    """
    for line in text.splitlines():
        if "[REMEMBER]:" in line:
            parts = line.split("[REMEMBER]:")
            if len(parts) > 1 and parts[1].strip():
                return parts[1].strip()
    return None


def build_prompt(user_message: str, memories: list) -> str:
    """
    Assemble the full prompt: system + memory context + user message.

    This is the injection point where Walrus memory enters the AI context.
    The AI sees past decisions as structured text before the user's message,
    allowing it to reference and reason about prior choices.

    Args:
        user_message: The current user input.
        memories:     List of recalled memory strings from Walrus.

    Returns:
        str: Complete prompt ready to send to any AI provider.
    """
    prompt = AGENT_STATE["system_prompt"]

    if memories:
        mem_block = "\n".join(f"  • {m}" for m in memories)
        prompt += f"\n\n── PAST DECISIONS FROM WALRUS MEMORY ──\n{mem_block}\n── END MEMORY ──"
    else:
        prompt += "\n\n── NO PRIOR MEMORY FOUND FOR THIS TOPIC ──"

    prompt += f"\n\nUser: {user_message}"
    return prompt


def agent_turn(user_message: str) -> dict:
    """
    Execute one full agent turn.

    Pipeline:
        1. Recall relevant memories from Walrus (if memory enabled)
        2. Build full prompt with memory injected
        3. Call AI provider (with automatic fallback)
        4. Parse response for [REMEMBER]: signal
        5. Write decision to Walrus if signal found (if memory enabled)

    Args:
        user_message: Raw input from the user.

    Returns:
        dict: Full turn result including response, provider, memory info.
    """
    AGENT_STATE["total_turns"] += 1
    result = {
        "response":       "",
        "provider":       "",
        "memories_found": [],
        "blob_saved":     False,
        "job_id":         None,
        "error":          None,
    }

    # ── Step 1: Recall ──
    memories = []
    if AGENT_STATE["memory_enabled"]:
        memories = walrus_recall(user_message)
    result["memories_found"] = memories

    # ── Step 2: Build prompt ──
    full_prompt = build_prompt(user_message, memories)

    # ── Step 3: Call AI ──
    try:
        response, provider = call_ai(full_prompt, AGENT_STATE["preferred_provider"])
        result["response"] = response
        result["provider"] = provider
    except RuntimeError as e:
        result["error"] = str(e)
        return result

    # ── Step 4 & 5: Remember ──
    if AGENT_STATE["memory_enabled"]:
        decision = extract_remember_signal(response)
        if decision:
            write_result = walrus_remember(decision)
            if write_result["status"] == "saved":
                result["blob_saved"] = True
                result["job_id"]     = write_result["job_id"]
                AGENT_STATE["session_blob_count"] += 1

    return result


print("✅ Agent Core loaded")
print(f"   Memory     : {'ON' if AGENT_STATE['memory_enabled'] else 'OFF'}")
print(f"   Provider   : {AGENT_STATE['preferred_provider'].upper()}")


✅ Agent Core loaded
   Memory     : ON
   Provider   : GEMINI


## Cell 6 — CLI Display Utilities

Formatting functions for the interactive CLI interface.
All visual output — headers, message bubbles, menus, tables — is defined here.

In [ ]:
# ─────────────────────────────────────────────
# CELL 6: CLI Display Utilities
# ─────────────────────────────────────────────

# ANSI color codes for terminal formatting
class C:
    RESET   = "\033[0m"
    BOLD    = "\033[1m"
    DIM     = "\033[2m"
    # Foreground
    BLUE    = "\033[94m"
    PURPLE  = "\033[95m"
    CYAN    = "\033[96m"
    GREEN   = "\033[92m"
    YELLOW  = "\033[93m"
    RED     = "\033[91m"
    WHITE   = "\033[97m"
    GRAY    = "\033[90m"
    # Background
    BG_BLUE = "\033[44m"
    BG_DARK = "\033[40m"


WIDTH = 62


def line(char="─", color=C.GRAY):
    print(f"{color}{char * WIDTH}{C.RESET}")


def banner():
    print()
    line("═", C.BLUE)
    title = "🧠  D E V W I T N E S S"
    pad   = (WIDTH - len(title)) // 2
    print(f"{C.BOLD}{C.BLUE}{' ' * pad}{title}{C.RESET}")
    sub   = "Persistent AI Memory · Powered by Walrus"
    pad2  = (WIDTH - len(sub)) // 2
    print(f"{C.GRAY}{' ' * pad2}{sub}{C.RESET}")
    line("═", C.BLUE)
    print()


def status_bar():
    """Show current agent status inline."""
    mem_icon  = f"{C.GREEN}●{C.RESET}" if AGENT_STATE["memory_enabled"] else f"{C.RED}●{C.RESET}"
    mem_label = f"{C.GREEN}MEMORY ON{C.RESET}" if AGENT_STATE["memory_enabled"] else f"{C.RED}MEMORY OFF{C.RESET}"
    prov      = AGENT_STATE["preferred_provider"].upper()
    blobs     = AGENT_STATE["session_blob_count"]
    turns     = AGENT_STATE["total_turns"]

    print(f"{C.GRAY}  {mem_icon} {mem_label}  "
          f"{C.CYAN}[ {prov} ]{C.RESET}  "
          f"{C.GRAY}Blobs this session: {C.YELLOW}{blobs}{C.RESET}  "
          f"{C.GRAY}Turns: {turns}{C.RESET}")
    line()


def print_user_message(text: str):
    print(f"\n  {C.BOLD}{C.BLUE}You{C.RESET}")
    print(f"  {C.BG_DARK}{C.WHITE} {text} {C.RESET}")


def print_ai_response(result: dict):
    """Print formatted AI response with memory metadata."""
    provider = result.get("provider", "Unknown")
    response = result.get("response", "")
    memories = result.get("memories_found", [])
    saved    = result.get("blob_saved", False)
    job_id   = result.get("job_id", "")
    error    = result.get("error")

    print(f"\n  {C.BOLD}{C.PURPLE}DevWitness{C.RESET} {C.GRAY}via {provider}{C.RESET}")

    if error:
        print(f"  {C.RED}❌ {error}{C.RESET}")
        return

    # Print response wrapped at WIDTH
    words = response.split()
    current_line = "  "
    for word in words:
        # Color special signals
        if "[RECALLED]" in word:
            word = f"{C.GREEN}[RECALLED]{C.RESET}"
        elif "[REMEMBER]:" in word:
            word = f"{C.YELLOW}[REMEMBER]:{C.RESET}"
        elif "[NO PRIOR MEMORY]" in word:
            word = f"{C.GRAY}[NO PRIOR MEMORY]{C.RESET}"

        if len(current_line) + len(word) + 1 > WIDTH:
            print(current_line)
            current_line = f"  {word} "
        else:
            current_line += f"{word} "

    if current_line.strip():
        print(current_line)

    # Memory metadata row
    print()
    if not AGENT_STATE["memory_enabled"]:
        print(f"  {C.RED}👻 Memory OFF — agent is flying blind{C.RESET}")
    elif memories:
        print(f"  {C.GREEN}🧠 {len(memories)} decision(s) recalled from Walrus{C.RESET}")
    else:
        print(f"  {C.GRAY}💭 No prior memory found for this topic{C.RESET}")

    if saved:
        short_id = job_id[:8] + "..." if len(str(job_id)) > 8 else str(job_id)
        print(f"  {C.YELLOW}💾 Decision written to Walrus · Job: {short_id}{C.RESET}")

    print()


def print_thinking():
    print(f"  {C.GRAY}⟳  Thinking...{C.RESET}")


def clear_thinking():
    # Move cursor up one line and clear it
    print("\033[A\033[K", end="")


print("✅ CLI Display Utilities loaded")

✅ CLI Display Utilities loaded


## Cell 7 — Settings Menus

All interactive settings submenus:
- Change AI provider
- Toggle memory on/off
- Edit system prompt
- View Walrus blob stats
- View current config

In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Settings Menus
# ─────────────────────────────────────────────


def menu_change_provider():
    """Submenu: select active AI provider."""
    print()
    line("─", C.CYAN)
    print(f"  {C.BOLD}{C.CYAN}SELECT AI PROVIDER{C.RESET}")
    line("─", C.CYAN)

    options = [
        ("1", "gemini",  "Gemini 2.0 Flash",     CONFIG["GEMINI_API_KEY"]),
        ("2", "groq",    "Groq / Llama-3.3-70B", CONFIG["GROQ_API_KEY"]),
        ("3", "openai",  "ChatGPT / GPT-4o Mini", CONFIG["OPENAI_API_KEY"]),
    ]

    for num, key, label, api_key in options:
        active  = " ◀ ACTIVE" if key == AGENT_STATE["preferred_provider"] else ""
        status  = f"{C.GREEN}✅{C.RESET}" if api_key else f"{C.RED}❌ no key{C.RESET}"
        print(f"  {C.YELLOW}[{num}]{C.RESET}  {label:<28} {status}{C.GREEN}{active}{C.RESET}")

    print(f"  {C.YELLOW}[0]{C.RESET}  Back")
    print()

    choice = input(f"  {C.BOLD}Choose provider: {C.RESET}").strip()
    mapping = {"1": "gemini", "2": "groq", "3": "openai"}

    if choice in mapping:
        key   = mapping[choice]
        label = PROVIDERS[key]["name"]
        if not CONFIG[PROVIDERS[key]["key"]]:
            print(f"  {C.RED}⚠️  No API key configured for {label}. Set it in Cell 2.{C.RESET}")
        else:
            AGENT_STATE["preferred_provider"] = key
            print(f"  {C.GREEN}✅ Switched to {label}{C.RESET}")
    elif choice != "0":
        print(f"  {C.RED}Invalid choice.{C.RESET}")


def menu_toggle_memory():
    """Toggle Walrus memory on or off."""
    current = AGENT_STATE["memory_enabled"]
    AGENT_STATE["memory_enabled"] = not current
    new_state = AGENT_STATE["memory_enabled"]

    if new_state:
        print(f"\n  {C.GREEN}🧠 Memory ENABLED — Walrus active{C.RESET}")
        print(f"  {C.GRAY}Agent will recall and write blobs this session.{C.RESET}")
    else:
        print(f"\n  {C.RED}👻 Memory DISABLED — Agent is now blind{C.RESET}")
        print(f"  {C.GRAY}No recall, no writes. Responses based on prompt only.{C.RESET}")


def menu_edit_prompt():
    """Submenu: view and edit the system prompt."""
    print()
    line("─", C.PURPLE)
    print(f"  {C.BOLD}{C.PURPLE}SYSTEM PROMPT EDITOR{C.RESET}")
    print(f"  {C.GRAY}The prompt is the agent's instruction set. Editing it")
    print(f"  changes how the agent thinks, remembers, and responds.{C.RESET}")
    line("─", C.PURPLE)
    print(f"\n  {C.YELLOW}[1]{C.RESET}  View current prompt")
    print(f"  {C.YELLOW}[2]{C.RESET}  Edit prompt (multiline)")
    print(f"  {C.YELLOW}[3]{C.RESET}  Reset to default")
    print(f"  {C.YELLOW}[0]{C.RESET}  Back")
    print()

    choice = input(f"  {C.BOLD}Choice: {C.RESET}").strip()

    if choice == "1":
        print()
        line("·", C.GRAY)
        for ln in AGENT_STATE["system_prompt"].splitlines():
            print(f"  {C.GRAY}{ln}{C.RESET}")
        line("·", C.GRAY)

    elif choice == "2":
        print(f"\n  {C.YELLOW}Enter new prompt. Type END on a new line when done:{C.RESET}\n")
        lines = []
        while True:
            ln = input("  ")
            if ln.strip().upper() == "END":
                break
            lines.append(ln)
        new_prompt = "\n".join(lines).strip()
        if new_prompt:
            AGENT_STATE["system_prompt"] = new_prompt
            print(f"\n  {C.GREEN}✅ Prompt updated. Agent behavior will change immediately.{C.RESET}")
        else:
            print(f"  {C.RED}Empty prompt — no changes made.{C.RESET}")

    elif choice == "3":
        AGENT_STATE["system_prompt"] = DEFAULT_SYSTEM_PROMPT
        print(f"\n  {C.GREEN}✅ Prompt reset to default.{C.RESET}")


def menu_walrus_stats():
    """Show Walrus blob statistics for this account."""
    print()
    line("─", C.GREEN)
    print(f"  {C.BOLD}{C.GREEN}WALRUS MEMORY STATS{C.RESET}")
    line("─", C.GREEN)

    acct = CONFIG["WALRUS_ACCOUNT_ID"]
    if acct:
        short = acct[:8] + "...." + acct[-4:] if len(acct) > 14 else acct
        print(f"  Account ID      : {C.CYAN}{short}{C.RESET}")
    else:
        print(f"  Account ID      : {C.RED}Not configured{C.RESET}")

    url = CONFIG["WALRUS_RELAYER_URL"]
    if url:
        domain = url.split("//")[-1].split("/")[0][:35]
        print(f"  Relayer         : {C.CYAN}{domain}{C.RESET}")
    else:
        print(f"  Relayer         : {C.RED}Not configured{C.RESET}")

    print(f"  Namespace       : {C.CYAN}{CONFIG['NAMESPACE']}{C.RESET}")
    print(f"  Session writes  : {C.YELLOW}{AGENT_STATE['session_blob_count']}{C.RESET}")

    print(f"\n  {C.GRAY}Fetching total blob count from Walrus...{C.RESET}")
    total = walrus_blob_count()
    if total >= 0:
        print(f"  Total blobs     : {C.GREEN}{total}{C.RESET}")
    else:
        print(f"  Total blobs     : {C.GRAY}Unavailable (check relayer){C.RESET}")

    line("─", C.GREEN)


def menu_settings():
    """Main settings menu."""
    while True:
        print()
        line("═", C.YELLOW)
        print(f"  {C.BOLD}{C.YELLOW}⚙  SETTINGS{C.RESET}")
        line("═", C.YELLOW)

        mem_status = f"{C.GREEN}ON{C.RESET}" if AGENT_STATE["memory_enabled"] else f"{C.RED}OFF{C.RESET}"
        prov       = AGENT_STATE["preferred_provider"].upper()

        print(f"  {C.YELLOW}[1]{C.RESET}  Change AI Provider        {C.GRAY}(current: {C.CYAN}{prov}{C.GRAY}){C.RESET}")
        print(f"  {C.YELLOW}[2]{C.RESET}  Toggle Memory             {C.GRAY}(current: {mem_status}{C.GRAY}){C.RESET}")
        print(f"  {C.YELLOW}[3]{C.RESET}  Edit System Prompt        {C.GRAY}(agent behavior){C.RESET}")
        print(f"  {C.YELLOW}[4]{C.RESET}  Walrus Blob Stats         {C.GRAY}(account info){C.RESET}")
        print(f"  {C.YELLOW}[0]{C.RESET}  Back to chat")
        print()

        choice = input(f"  {C.BOLD}Choose: {C.RESET}").strip()

        if   choice == "1": menu_change_provider()
        elif choice == "2": menu_toggle_memory()
        elif choice == "3": menu_edit_prompt()
        elif choice == "4": menu_walrus_stats()
        elif choice == "0": break
        else: print(f"  {C.RED}Invalid choice.{C.RESET}")


print("✅ Settings Menus loaded")

✅ Settings Menus loaded


## Cell 8 — Main CLI Loop

The interactive agent loop. Commands available during chat:

| Command | Action |
|---|---|
| `/settings` or `/s` | Open settings menu |
| `/memory` or `/m` | Toggle memory on/off instantly |
| `/provider` or `/p` | Quick provider switch |
| `/clear` | Clear session (memory persists on Walrus) |
| `/status` | Show current agent status |
| `/help` | Show all commands |
| `exit` / `quit` | Exit the agent |

**Run this cell to start the agent.**

In [9]:
# ─────────────────────────────────────────────
# CELL 8: Main CLI Loop — RUN THIS TO START
# ─────────────────────────────────────────────


def print_help():
    print()
    line("─", C.CYAN)
    print(f"  {C.BOLD}{C.CYAN}DEVWITNESS COMMANDS{C.RESET}")
    line("─", C.CYAN)
    cmds = [
        ("/settings, /s",  "Open settings menu"),
        ("/memory,   /m",  "Toggle memory on/off"),
        ("/provider, /p",  "Quick provider switch"),
        ("/clear",         "Clear session chat (Walrus memory persists)"),
        ("/status",        "Show current agent status"),
        ("/help,     /h",  "Show this help"),
        ("exit / quit",    "Exit DevWitness"),
    ]
    for cmd, desc in cmds:
        print(f"  {C.YELLOW}{cmd:<18}{C.RESET} {C.GRAY}{desc}{C.RESET}")
    line("─", C.CYAN)


def print_status():
    print()
    line("─", C.BLUE)
    print(f"  {C.BOLD}{C.BLUE}AGENT STATUS{C.RESET}")
    line("─", C.BLUE)
    mem = f"{C.GREEN}ENABLED ● {C.RESET}" if AGENT_STATE["memory_enabled"] else f"{C.RED}DISABLED ○{C.RESET}"
    print(f"  Memory           : {mem}")
    print(f"  Active Provider  : {C.CYAN}{AGENT_STATE['preferred_provider'].upper()}{C.RESET}")
    print(f"  Session Blobs    : {C.YELLOW}{AGENT_STATE['session_blob_count']}{C.RESET}")
    print(f"  Total Turns      : {C.GRAY}{AGENT_STATE['total_turns']}{C.RESET}")
    print(f"  Namespace        : {C.GRAY}{CONFIG['NAMESPACE']}{C.RESET}")
    line("─", C.BLUE)


def run_devwitness():
    """Main interactive loop."""

    banner()
    status_bar()

    print(f"  {C.GRAY}Type a technical question to start.")
    print(f"  Type /help for commands.{C.RESET}")
    print()

    while True:
        try:
            # ── Prompt ──
            mem_indicator = f"{C.GREEN}🧠{C.RESET}" if AGENT_STATE["memory_enabled"] else f"{C.RED}👻{C.RESET}"
            prov          = C.CYAN + AGENT_STATE["preferred_provider"].upper() + C.RESET
            user_input    = input(f"  {mem_indicator} {prov} {C.BOLD}▶{C.RESET}  ").strip()

            if not user_input:
                continue

            # ── Commands ──
            cmd = user_input.lower()

            if cmd in ("exit", "quit", "q"):
                print(f"\n  {C.BLUE}Thanks for using DevWitness. Your memories live on Walrus.{C.RESET}")
                print(f"  {C.GRAY}Session blobs written: {AGENT_STATE['session_blob_count']}{C.RESET}\n")
                break

            elif cmd in ("/settings", "/s"):
                menu_settings()
                print()
                status_bar()
                continue

            elif cmd in ("/memory", "/m"):
                menu_toggle_memory()
                continue

            elif cmd in ("/provider", "/p"):
                menu_change_provider()
                continue

            elif cmd == "/clear":
                print(f"\n  {C.YELLOW}Session cleared.{C.RESET}")
                print(f"  {C.GRAY}Walrus memory persists — start a new session to prove it.{C.RESET}\n")
                AGENT_STATE["total_turns"] = 0
                AGENT_STATE["session_blob_count"] = 0
                status_bar()
                continue

            elif cmd == "/status":
                print_status()
                continue

            elif cmd in ("/help", "/h"):
                print_help()
                continue

            # ── Agent turn ──
            print_user_message(user_input)
            print_thinking()

            result = agent_turn(user_input)

            clear_thinking()
            print_ai_response(result)
            line()

        except KeyboardInterrupt:
            print(f"\n\n  {C.YELLOW}Interrupted. Type 'exit' to quit cleanly.{C.RESET}\n")
            continue

        except Exception as e:
            print(f"  {C.RED}Unexpected error: {e}{C.RESET}")
            continue


# ── LAUNCH ──
run_devwitness()


══════════════════════════════════════════════════════════════
                    🧠  D E V W I T N E S S
           Persistent AI Memory · Powered by Walrus
══════════════════════════════════════════════════════════════

  ● MEMORY ON  [ GEMINI ]  Blobs this session: 0  Turns: 0
──────────────────────────────────────────────────────────────
  Type a technical question to start.
  Type /help for commands.


  You
   Hi 
  ⟳  Thinking...
  [recall] returned 9 memories
  ⚠️  Gemini 2.0 Flash failed: 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'Request had invalid aut
  ↪️  Trying next provider...

  DevWitness via Groq / Llama-3.3-70B
  [NO PRIOR MEMORY] on this specific conversation topic, as 
  the previous memory context seems to be about various 
  technical decisions related to coding languages and 
  libraries. It appears there was a discussion about using 
  Python, TypeScript, HTML, and Pandas for different tasks. 
  However, I don't see any direct relation to a gree